In [3]:
import pandas as pd

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [7]:
stk_data=pd.read_csv("Tatacoffee13_21.csv",parse_dates=['Date'],index_col='Date')

In [9]:
stk_data

,Open,High,Low,Close
Date,,,,
2013-01-01,1410.60,1427.90,1408.30,1415.10
2013-01-02,1421.00,1626.60,1416.15,1607.40
2013-01-03,1632.55,1673.90,1613.05,1626.20
2013-01-04,1627.75,1627.75,1574.60,1579.05
2013-01-07,1580.00,1639.50,1565.50,1595.65
...,...,...,...,...
2021-12-22,202.90,207.80,201.35,205.00
2021-12-23,206.00,206.85,202.05,202.95
2021-12-24,203.90,203.90,199.35,201.00


In [11]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2225, 4)


In [12]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [13]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

1780
X_train length: (1780, 4)
X_test length: (445, 4)
y_train length: (1780, 4)
y_test length: (445, 4)


In [70]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}
performance

{'Model': [], 'RMSE': [], 'MaPe': [], 'Lag': [], 'Test': []}

In [72]:
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs = 28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    from statsmodels.tsa.api import VAR
    for i in [1,2,3,4,5,6,7,8,9,10]:
        model = VAR(train)
        results = model.fit(i)
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
    x = model.select_order(maxlags=12)
    order=x.selected_orders["aic"]
    result = model.fit(order)
    #result.summary()
    lagged_Values = train.values[-order:]
    pred = result.forecast(y=lagged_Values,steps=28) 
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    from sklearn.metrics import root_mean_squared_error
    rmse= root_mean_squared_error(test,pred)
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,result,pred


In [90]:
listt=["Close","High","Low","Open"]

In [92]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Low', 'Open']
Order = 1
AIC:  -41.57633250372654
BIC:  -41.52447100279185

Order = 2
AIC:  -41.89965675983841
BIC:  -41.80627099960227

Order = 3
AIC:  -41.896681081226745
BIC:  -41.761739857287175

Order = 4
AIC:  -41.89556182371679
BIC:  -41.719033893533286

Order = 5
AIC:  -41.92461661731271
BIC:  -41.70647070014222

Order = 6
AIC:  -42.00676954931913
BIC:  -41.746974326151836

Order = 7
AIC:  -42.06304859245056
BIC:  -41.76157270594553

Order = 8
AIC:  -42.058911836473946
BIC:  -41.71572389089468

Order = 9
AIC:  -42.075067742123125
BIC:  -41.69013630327294

Order = 10
AIC:  -42.09630316989201
BIC:  -41.669596765049306



In [94]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High]",0.010948,0.103428,7,28
1,"[Close, High, Low]",0.010290,0.096243,12,28
2,"[Close, High, Low, Open]",0.009534,0.088105,11,28
